In [28]:
import os
import platform
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl

from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from multiprocessing import cpu_count

In [29]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
pl.seed_everything(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Global seed set to 42


In [30]:
SEQ_LEN = 200
# SEQ_LEN = 1317
BATCH_SIZE = 4
EMBED_DIM = 128
DROP_OUT = 0.9
NUM_WORKERS = 0 if platform.system() == 'Windows' else cpu_count()

Q_is_S = True
print("os:{}, num-workers:{}".format(platform.system(), NUM_WORKERS))

os:Linux, num-workers:16


In [31]:
df_train = pd.read_csv('dataset/algebra_2006_2007/train.csv', low_memory=False, encoding="ISO-8859-1")
df_val = pd.read_csv('dataset/algebra_2006_2007/test.csv', low_memory=False, encoding="ISO-8859-1")
df_train.head()

,user_id,q_idx,s_idx,q_type,q_diff,ms_first_response,correct
0,271qgzl01euj,13359,10,7,0.883450,0.120948,1
1,271qgzl01euj,13359,11,7,0.883450,0.007170,1
2,271qgzl01euj,29143,12,7,0.857143,0.030237,0
3,271qgzl01euj,29143,13,7,0.857143,0.002805,1
4,271qgzl01euj,29144,12,7,1.000000,0.003741,1


In [32]:
key_user = 'user_id'
key_q = 'q_idx'
key_s = 's_idx'
key_qtype = 'q_type'
key_ms = 'ms_first_response'
# key_attempt = 'attempt_count'
key_correct = 'correct'
key_diff = 'q_diff'

In [33]:
# 我们需要将数据进行预处理，每个学生的学习记录利用group by合并为序列。
def generate_group_by_df(df):
    KEY = key_s if Q_is_S else key_q
    group = df.groupby([key_user]).apply(lambda r: (
                r[KEY].values,
                r[key_s].values,
                r[key_qtype].values,
                r[key_diff].values,
                r[key_ms].values,
                
                r[key_correct].values                                                                                                                                                                                                                                      
                ))
    return group

In [34]:
train = generate_group_by_df(df_train) 
val = generate_group_by_df(df_val)
train.head()

user_id
0I891Gg     ([125, 125, 132, 134, 131, 125, 129, 136, 130,...
171017OL    ([0, 0, 3, 0, 0, 1, 2, 1, 2, 1, 5, 6, 8, 7, 8,...
171051xl    ([12, 18, 10, 11, 10, 14, 11, 10, 14, 11, 10, ...
1710gLX8    ([8, 8, 8, 8, 7, 7, 9, 9, 9, 5, 0, 0, 1, 2, 1,...
1710nc7l    ([327, 315, 316, 317, 318, 319, 321, 320, 322,...
dtype: object

In [35]:
N_QUERY_FEATURES = len(train.iloc[0])-1
print("N_QUERY_FEATURES:{}".format(N_QUERY_FEATURES))

N_QUERY_FEATURES:5


In [36]:
N_QUESTION = len(df_train[key_q].unique()) + len(df_val[key_q].unique())
N_SKILL = len(df_train[key_s].unique()) + len(df_val[key_s].unique())
N_QUESTION_TYPE = len(df_train[key_qtype].unique()) + len(df_val[key_qtype].unique())
N_QUESTION, N_SKILL, N_QUESTION_TYPE

(99787, 2556, 304)

In [37]:

from data_loader.saintdataset import SAINTDataset

N_Q_OR_S = N_SKILL if Q_is_S else N_QUESTION


train_dataset = SAINTDataset(train, N_Q_OR_S, SEQ_LEN)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

val_dataset = SAINTDataset(val, N_Q_OR_S, SEQ_LEN)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("train:{}, test:{}".format(len(train_dataset), len(val_dataset)))

train:1068, test:268


In [38]:
import warnings
warnings.filterwarnings('ignore')

from model.saint2 import SAINTModule


model = SAINTModule(
    dim_model=EMBED_DIM,
    num_en=6,
    num_de=6,
    heads_en=8,
    heads_de=8,
    total_ex=N_Q_OR_S,
    total_cat=N_QUESTION_TYPE,
    total_in=2,
    seq_len=SEQ_LEN
)
checkpoint_callback = pl.callbacks.ModelCheckpoint(save_top_k=1, verbose=True, monitor='v_auc', mode='max')

patience = 6 if N_QUERY_FEATURES == 6 else 3
# sakt.train_dataloader
trainer = pl.Trainer(
    gpus=1, 
    max_epochs=200, 
    auto_lr_find=True, 
    callbacks=[checkpoint_callback, EarlyStopping(monitor="v_auc", mode="max", patience=6)]
)
print("patience:{}".format(patience))

GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


patience:3


In [39]:
trainer.fit(model=model, train_dataloaders=train_dataloader,val_dataloaders=val_dataloader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type              | Params
--------------------------------------------
0 | loss  | BCEWithLogitsLoss | 0     
1 | model | saint             | 3.8 M 
--------------------------------------------
3.8 M     Trainable params
0         Non-trainable params
3.8 M     Total params
15.164    Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Epoch 0, global step 267: 'v_auc' reached 0.55900 (best 0.55900), saving model to '/home/czy/KT/BRIKT/lightning_logs/version_111/checkpoints/epoch=0-step=267.ckpt' as top 1


...

Epoch 20, global step 5607: 'v_auc' was not in top 1


Validation: 0it [00:00, ?it/s]

Epoch 21, global step 5874: 'v_auc' was not in top 1


Validation: 0it [00:00, ?it/s]

Epoch 22, global step 6141: 'v_auc' was not in top 1
